# Module 2: Data Cleaning & Audit

## 2.0 · Configuration

In [ ]:
PROJECT_DIR = "CounterShield"
DATASETS = ["bot_iot", "cic_ids2017", "unsw_nb15"]

DROP_CONFLICTING     = True
UNSW_DEDUP_TEST      = False
RARE_CLASS_THRESHOLD = 100

CFG = {
    "bot_iot": {
        "identifiers": ["pkSeqID", "stime", "ltime", "seq", "saddr", "daddr", "smac", "dmac", "soui", "doui", "sco", "dco"],
        "raw_label_cols": ["attack", "category", "subcategory"],
        "subclass_col": "subcategory",
        "categorical": ["flgs", "proto", "state"],
        "sentinel_fill": {"sport": -1, "dport": -1},
        "dedup_group": None,
    },
    "cic_ids2017": {
        "identifiers": ["Flow ID", "Source IP", "Src IP", "Destination IP", "Dst IP", "Source Port", "Src Port", "Destination Port", "Dst Port", "Timestamp"],
        "raw_label_cols": ["Label", "Attack"],
        "categorical": [],
        "sentinel_fill": {},
        "dedup_group": None,
    },
    "unsw_nb15": {
        "identifiers": ["id", "srcip", "dstip", "sport", "stime", "ltime"],
        "raw_label_cols": ["label", "attack_cat"],
        "categorical": ["proto", "service", "state"],
        "sentinel_fill": {},
        "dedup_group": "split",
        "dedup_skip_groups": [] if UNSW_DEDUP_TEST else ["test"],
    },
}
OPTS = {"drop_conflicting": DROP_CONFLICTING}

### Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import sys, datetime as dt
try:
    from IPython.display import display
except ImportError:
    display = print

import os, re, gc, json, time, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd

LABEL_COLS = ["y_binary", "y_class", "y_family", "y_subclass"]
META_COLS = ["source_dataset", "split", "source_file"]

def feature_cols(df):
    return [c for c in df.columns if c not in LABEL_COLS + META_COLS]

def row_hash(df, cols):
    return pd.util.hash_pandas_object(df[cols], index=False).to_numpy()

def sha256_file(path, block=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()

def human(n):
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024: return f"{n:,.1f} {u}"
        n /= 1024
    return f"{n:,.1f} PB"

def _norm_str(s):
    s = s.astype(str).str.strip().str.lower()
    return s.replace({"nan": "unknown", "": "unknown", "none": "unknown", "<na>": "unknown"})

def _norm_label(x):
    s = re.sub(r"[^a-z0-9]+", "_", str(x).strip().lower()).strip("_")
    return "benign" if s in {"benign", "normal", "0", "", "nan", "none"} else s

def clean_dataset(df, key, cfg, opts):
    df = df.copy()
    log = {"dataset": key, "rows_in": int(len(df)), "cols_in": int(df.shape[1]), "steps": []}
    def step(name, **info):
        log["steps"].append({"step": name, "rows_after": int(len(df)), "cols_after": int(df.shape[1]), **info})

    ids = [c for c in cfg["identifiers"] if c in df.columns]
    df = df.drop(columns=ids); step("drop_identifier_columns", columns=ids)
    if cfg.get("subclass_col") in df.columns:
        df["y_subclass"] = df[cfg["subclass_col"]].map(_norm_label).astype("category")
    raw_lab = [c for c in cfg["raw_label_cols"] if c in df.columns]
    df = df.drop(columns=raw_lab); step("drop_raw_label_columns", columns=raw_lab, note="kept as y_binary / y_class / y_family" + (" / y_subclass" if "y_subclass" in df.columns else ""))
    for c in ("y_class", "y_family"):
        df[c] = df[c].astype(str).astype("category")
    df["y_binary"] = df["y_binary"].astype("int8")

    cats = [c for c in cfg["categorical"] if c in df.columns]
    for c in cats:
        df[c] = _norm_str(df[c]).astype("category")
    converted, extra_cats = [], []
    for c in feature_cols(df):
        if c in cats or pd.api.types.is_numeric_dtype(df[c]): continue
        num = pd.to_numeric(df[c], errors="coerce")
        if num.notna().mean() >= 0.99:
            df[c] = num; converted.append(c)
        else:
            df[c] = _norm_str(df[c]).astype("category"); extra_cats.append(c)
    log["categorical_columns"] = cats + extra_cats
    step("standardise_types", categorical=cats + extra_cats, converted_to_numeric=converted)

    num_cols = [c for c in feature_cols(df) if pd.api.types.is_numeric_dtype(df[c])]
    n_inf = {c: int(np.isinf(df[c].to_numpy(dtype="float64", na_value=np.nan)).sum()) for c in num_cols}
    n_inf = {c: v for c, v in n_inf.items() if v}
    if n_inf:
        df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
    step("infinities_to_nan", counts=n_inf)

    feats = feature_cols(df)
    all_nan = [c for c in feats if df[c].isna().all()]
    const = [c for c in feats if c not in all_nan and df[c].nunique(dropna=False) <= 1]
    df = df.drop(columns=all_nan + const)
    step("drop_empty_and_constant_columns", all_nan=all_nan, constant=const)

    fill = {}
    for c in cfg.get("sentinel_fill", {}):
        if c in df.columns and df[c].isna().any():
            fill[c] = int(df[c].isna().sum()); df[c] = df[c].fillna(cfg["sentinel_fill"][c])
    step("sentinel_fill", filled=fill, values=cfg.get("sentinel_fill", {}))
    feats = feature_cols(df)
    nan_rows = df[feats].isna().any(axis=1)
    nan_by_col = {c: int(v) for c, v in df.loc[nan_rows, feats].isna().sum().items() if v}
    df = df.loc[~nan_rows]
    step("drop_rows_with_missing_features", rows_dropped=int(nan_rows.sum()), by_column=nan_by_col)

    neg = {}
    for c in feature_cols(df):
        if pd.api.types.is_numeric_dtype(df[c]) and c not in cfg.get("sentinel_fill", {}):
            n = int((df[c] < 0).sum())
            if n: neg[c] = n
    log["negative_value_columns"] = neg
    step("negative_value_report", policy="report_only", columns=neg)

    feats = feature_cols(df)
    groups = df[cfg["dedup_group"]].astype(str) if cfg.get("dedup_group") else pd.Series("all", index=df.index)
    keep = np.ones(len(df), dtype=bool)
    dup_info = {}
    for g in groups.unique():
        m = (groups == g).to_numpy()
        if g in cfg.get("dedup_skip_groups", []):
            sub = df.loc[m]
            hf = row_hash(sub, feats); hl = row_hash(sub, feats + ["y_class"])
            dup_info[g] = {"skipped": True, "exact_duplicates_present": int(pd.Series(hl).duplicated().sum())}
            continue
        idx = np.flatnonzero(m)
        sub = df.iloc[idx]
        hf = row_hash(sub, feats)
        tmp = pd.DataFrame({"h": hf, "y": sub["y_class"].astype(str).to_numpy()})
        nlab = tmp.groupby("h")["y"].nunique()
        conflict = tmp["h"].isin(nlab.index[nlab > 1]).to_numpy() if opts["drop_conflicting"] else np.zeros(len(tmp), bool)
        hl = row_hash(sub, feats + ["y_class"])
        exact = pd.Series(hl).duplicated(keep="first").to_numpy() & ~conflict
        keep[idx[conflict | exact]] = False
        dup_info[g] = {"conflicting_rows_removed": int(conflict.sum()),
                       "conflicting_feature_groups": int((nlab > 1).sum()),
                       "exact_duplicates_removed": int(exact.sum())}
    df = df.loc[keep]
    step("remove_duplicates", by_group=dup_info)

    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = df[c].astype("float32")
    df = df.reset_index(drop=True)
    log["rows_out"], log["cols_out"] = int(len(df)), int(df.shape[1])
    log["duplicates"] = dup_info
    return df, log

def missing_audit(raw, clean, key, log):
    rows = []
    fill = next(s for s in log["steps"] if s["step"] == "sentinel_fill")["filled"]
    dropped = next(s for s in log["steps"] if s["step"] == "drop_empty_and_constant_columns")
    by_col = next(s for s in log["steps"] if s["step"] == "drop_rows_with_missing_features")["by_column"]
    infs = next(s for s in log["steps"] if s["step"] == "infinities_to_nan")["counts"]
    for c in raw.columns:
        s = raw[c]
        n_nan = int(s.isna().sum())
        n_inf = int(np.isinf(s.to_numpy(dtype="float64", na_value=np.nan)).sum()) if pd.api.types.is_numeric_dtype(s) else 0
        if c in dropped["all_nan"]: action = "column dropped (all missing)"
        elif c in fill: action = f"filled with sentinel ({fill[c]:,} values)"
        elif c in by_col: action = f"rows dropped ({by_col[c]:,})"
        elif c not in clean.columns: action = "column removed (identifier / raw label / constant)"
        else: action = "none needed" if (n_nan + n_inf) == 0 else "handled"
        rows.append({"dataset": key, "column": c, "dtype": str(s.dtype), "missing_raw": n_nan, "inf_raw": n_inf,
                     "pct_missing_raw": round(100 * (n_nan + n_inf) / max(len(raw), 1), 4),
                     "missing_clean": int(clean[c].isna().sum()) if c in clean.columns else None, "action": action})
    return pd.DataFrame(rows)

def label_distribution(df, key, stage):
    split = df["split"].astype(str) if "split" in df.columns else pd.Series("all", index=df.index)
    out = (pd.DataFrame({"split": split, "y_family": df["y_family"].astype(str), "y_class": df["y_class"].astype(str),
                         "y_binary": df["y_binary"].astype(int)})
             .value_counts().rename("rows").reset_index())
    out.insert(0, "stage", stage); out.insert(0, "dataset", key)
    return out

def label_checks(raw, clean, key, cfg, rare_threshold):
    chk = []
    def add(name, n, detail=""):
        chk.append({"dataset": key, "check": name, "violations": int(n), "status": "PASS" if n == 0 else "FLAG", "detail": detail})
    add("no missing labels (clean)", clean[["y_binary", "y_class", "y_family"]].isna().any(axis=1).sum())
    add("y_binary in {0,1}", (~clean["y_binary"].isin([0, 1])).sum())
    add("y_binary agrees with y_class (clean)", ((clean["y_class"].astype(str) != "benign").astype(int) != clean["y_binary"]).sum())
    if key == "bot_iot" and {"attack", "category"} <= set(raw.columns):
        add("raw 'attack' agrees with raw 'category'", ((raw["category"].astype(str).str.strip().str.lower() != "normal").astype(int) != pd.to_numeric(raw["attack"], errors="coerce")).sum())
    if key == "unsw_nb15" and {"label", "attack_cat"} <= set(raw.columns):
        add("raw 'label' agrees with raw 'attack_cat'", ((raw["attack_cat"].map(_norm_label) != "benign").astype(int) != pd.to_numeric(raw["label"], errors="coerce")).sum())
    lost = sorted(set(raw["y_class"].astype(str)) - set(clean["y_class"].astype(str)))
    add("no class lost during cleaning", len(lost), ", ".join(lost))
    vc = clean["y_class"].astype(str).value_counts()
    rare = vc[vc < rare_threshold]
    add(f"classes with < {rare_threshold} rows", len(rare), ", ".join(f"{k} ({v})" for k, v in rare.items()))
    add("both benign and attack present", 0 if clean["y_binary"].nunique() == 2 else 1)
    return pd.DataFrame(chk), {"imbalance_ratio_class": round(float(vc.max() / vc.min()), 1),
                               "attack_rate": round(float(clean["y_binary"].mean()), 4),
                               "n_classes": int(len(vc))}

def validate(clean, key, cfg, log):
    feats = feature_cols(clean)
    num = [c for c in feats if pd.api.types.is_numeric_dtype(clean[c])]
    res = []
    def add(name, ok, detail=""):
        res.append({"dataset": key, "check": name, "passed": bool(ok), "detail": detail})
    add("no missing values in features", not clean[feats].isna().any().any())
    add("no infinite values", not np.isinf(clean[num].to_numpy(dtype="float64")).any() if num else True)
    groups = clean[cfg["dedup_group"]].astype(str) if cfg.get("dedup_group") else pd.Series("all", index=clean.index)
    ndup = 0
    for g in groups.unique():
        if g in cfg.get("dedup_skip_groups", []): continue
        sub = clean.loc[groups == g]
        ndup += int(pd.Series(row_hash(sub, feats + ["y_class"])).duplicated().sum())
    add("no exact duplicate rows (dedup groups)", ndup == 0, f"{ndup} found")
    bad_types = [c for c in feats if not (pd.api.types.is_numeric_dtype(clean[c]) or isinstance(clean[c].dtype, pd.CategoricalDtype))]
    add("features are numeric or categorical", not bad_types, ", ".join(bad_types))
    add("no identifier columns left", not [c for c in cfg["identifiers"] if c in clean.columns])
    add("label columns present", all(c in clean.columns for c in ["y_binary", "y_class", "y_family"]))
    add("two target classes present", clean["y_binary"].nunique() == 2)
    add("row count matches cleaning log", len(clean) == log["rows_out"])
    return pd.DataFrame(res)

BASE       = Path("/content/drive/MyDrive") / PROJECT_DIR
IN_DIR     = BASE / "processed"
OUT_DIR    = BASE / "processed" / "clean"
AUDIT_DIR  = BASE / "results" / "module2"
CONFIG_DIR = BASE / "configs"
for p in (OUT_DIR, AUDIT_DIR, CONFIG_DIR): p.mkdir(parents=True, exist_ok=True)

RAW = {}
for k in DATASETS:
    f = IN_DIR / f"{k}.parquet"
    RAW[k] = pd.read_parquet(f)
    print(f"{k:<12} {RAW[k].shape[0]:>10,} rows x {RAW[k].shape[1]:>3} cols   ({human(f.stat().st_size)})")
CLEAN, LOGS = {}, {}

def show_log(log):
    rows = []
    for s in log["steps"]:
        info = {k: v for k, v in s.items() if k not in ("step", "rows_after", "cols_after")}
        rows.append({"step": s["step"], "rows": f"{s['rows_after']:,}", "cols": s["cols_after"],
                     "details": json.dumps(info, default=str)[:180]})
    display(pd.DataFrame(rows))
    print(f"{log['dataset']}: {log['rows_in']:,} -> {log['rows_out']:,} rows ({log['rows_in']-log['rows_out']:,} removed), "
          f"{log['cols_in']} -> {log['cols_out']} columns")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bot_iot         911,828 rows x  40 cols   (35.0 MB)
cic_ids2017   2,827,876 rows x  85 cols   (258.2 MB)
unsw_nb15       257,673 rows x  50 cols   (15.6 MB)


## 2.1 · BoT-IoT cleaning

In [ ]:
CLEAN["bot_iot"], LOGS["bot_iot"] = clean_dataset(RAW["bot_iot"], "bot_iot", CFG["bot_iot"], OPTS)
show_log(LOGS["bot_iot"])

,step,rows,cols,details
0,drop_identifier_columns,"911,828",28,"{""columns"": [""pkSeqID"", ""stime"", ""ltime"", ""seq..."
1,drop_raw_label_columns,"911,828",26,"{""columns"": [""attack"", ""category"", ""subcategor..."
2,standardise_types,"911,828",26,"{""categorical"": [""flgs"", ""proto"", ""state""], ""c..."
3,infinities_to_nan,"911,828",26,"{""counts"": {}}"
4,drop_empty_and_constant_columns,"911,828",26,"{""all_nan"": [], ""constant"": []}"
5,sentinel_fill,"911,828",26,"{""filled"": {""sport"": 564, ""dport"": 564}, ""valu..."
6,drop_rows_with_missing_features,"911,828",26,"{""rows_dropped"": 0, ""by_column"": {}}"
7,negative_value_report,"911,828",26,"{""policy"": ""report_only"", ""columns"": {}}"
8,remove_duplicates,"903,922",26,"{""by_group"": {""all"": {""conflicting_rows_remove..."


bot_iot: 911,828 -> 903,922 rows (7,906 removed), 40 -> 26 columns


## 2.2 · CICIDS-2017 cleaning

In [ ]:
CLEAN["cic_ids2017"], LOGS["cic_ids2017"] = clean_dataset(RAW["cic_ids2017"], "cic_ids2017", CFG["cic_ids2017"], OPTS)
show_log(LOGS["cic_ids2017"])

,step,rows,cols,details
0,drop_identifier_columns,"2,827,876",84,"{""columns"": [""Destination Port""]}"
1,drop_raw_label_columns,"2,827,876",83,"{""columns"": [""Label""], ""note"": ""kept as y_bina..."
2,standardise_types,"2,827,876",83,"{""categorical"": [], ""converted_to_numeric"": []}"
3,infinities_to_nan,"2,827,876",83,"{""counts"": {}}"
4,drop_empty_and_constant_columns,"2,827,876",75,"{""all_nan"": [], ""constant"": [""Bwd PSH Flags"", ..."
5,sentinel_fill,"2,827,876",75,"{""filled"": {}, ""values"": {}}"
6,drop_rows_with_missing_features,"2,827,876",75,"{""rows_dropped"": 0, ""by_column"": {}}"
7,negative_value_report,"2,827,876",75,"{""policy"": ""report_only"", ""columns"": {""Flow Du..."
8,remove_duplicates,"2,198,595",75,"{""by_group"": {""all"": {""conflicting_rows_remove..."


cic_ids2017: 2,827,876 -> 2,198,595 rows (629,281 removed), 85 -> 75 columns


## 2.3 · UNSW-NB15 cleaning

In [ ]:
CLEAN["unsw_nb15"], LOGS["unsw_nb15"] = clean_dataset(RAW["unsw_nb15"], "unsw_nb15", CFG["unsw_nb15"], OPTS)
show_log(LOGS["unsw_nb15"])

,step,rows,cols,details
0,drop_identifier_columns,"257,673",49,"{""columns"": [""id""]}"
1,drop_raw_label_columns,"257,673",47,"{""columns"": [""label"", ""attack_cat""], ""note"": ""..."
2,standardise_types,"257,673",47,"{""categorical"": [""proto"", ""service"", ""state""],..."
3,infinities_to_nan,"257,673",47,"{""counts"": {}}"
4,drop_empty_and_constant_columns,"257,673",47,"{""all_nan"": [], ""constant"": []}"
5,sentinel_fill,"257,673",47,"{""filled"": {}, ""values"": {}}"
6,drop_rows_with_missing_features,"257,673",47,"{""rows_dropped"": 0, ""by_column"": {}}"
7,negative_value_report,"257,673",47,"{""policy"": ""report_only"", ""columns"": {}}"
8,remove_duplicates,"181,600",47,"{""by_group"": {""test"": {""skipped"": true, ""exact..."


unsw_nb15: 257,673 -> 181,600 rows (76,073 removed), 50 -> 47 columns


## 2.4 · Missing-value audit

In [ ]:
miss = pd.concat([missing_audit(RAW[k], CLEAN[k], k, LOGS[k]) for k in DATASETS], ignore_index=True)
miss.to_csv(AUDIT_DIR / "missing_values.csv", index=False)
issues = miss[(miss.missing_raw + miss.inf_raw) > 0]
print(f"Columns with missing or infinite values in the Module 1 output: {len(issues)}")
display(issues if len(issues) else "none")
print("Missing values left after cleaning:", int(miss.missing_clean.fillna(0).sum()))

Columns with missing or infinite values in the Module 1 output: 8


,dataset,column,dtype,missing_raw,inf_raw,pct_missing_raw,missing_clean,action
5,bot_iot,sport,float32,564,0,0.0619,0.0,filled with sentinel (564 values)
7,bot_iot,dport,float32,564,0,0.0619,0.0,filled with sentinel (564 values)
16,bot_iot,smac,float32,911828,0,100.0000,NaN,column removed (identifier / raw label / const...
17,bot_iot,dmac,float32,911828,0,100.0000,NaN,column removed (identifier / raw label / const...
21,bot_iot,soui,float32,911828,0,100.0000,NaN,column removed (identifier / raw label / const...
22,bot_iot,doui,float32,911828,0,100.0000,NaN,column removed (identifier / raw label / const...
23,bot_iot,sco,float32,911828,0,100.0000,NaN,column removed (identifier / raw label / const...
24,bot_iot,dco,float32,911828,0,100.0000,NaN,column removed (identifier / raw label / const...


Missing values left after cleaning: 0


## 2.5 · Duplicate & data-integrity audit

In [ ]:
integ = []
for k in DATASETS:
    L = LOGS[k]; st = {s["step"]: s for s in L["steps"]}
    dup = L["duplicates"]
    integ.append({
        "dataset": k,
        "rows_in": L["rows_in"], "rows_out": L["rows_out"],
        "rows_removed_pct": round(100 * (L["rows_in"] - L["rows_out"]) / L["rows_in"], 2),
        "exact_duplicates_removed": sum(v.get("exact_duplicates_removed", 0) for v in dup.values()),
        "conflicting_rows_removed": sum(v.get("conflicting_rows_removed", 0) for v in dup.values()),
        "rows_with_missing_dropped": st["drop_rows_with_missing_features"]["rows_dropped"],
        "identifier_cols_dropped": len(st["drop_identifier_columns"]["columns"]),
        "constant_cols_dropped": len(st["drop_empty_and_constant_columns"]["constant"]),
        "empty_cols_dropped": len(st["drop_empty_and_constant_columns"]["all_nan"]),
        "cols_with_negatives": len(L["negative_value_columns"]),
        "features_out": len(feature_cols(CLEAN[k])),
        "memory_MB": round(CLEAN[k].memory_usage(deep=True).sum() / 2**20, 1),
    })
integ = pd.DataFrame(integ)
display(integ)

u = CLEAN["unsw_nb15"]; fu = feature_cols(u)
h = pd.Series(row_hash(u, fu))
tr, te = h[(u.split == "train").to_numpy()], h[(u.split == "test").to_numpy()]
overlap = int(te.isin(set(tr)).sum())
test_dups = LOGS["unsw_nb15"]["duplicates"].get("test", {})
print(f"UNSW-NB15: {overlap:,} of {len(te):,} test rows ({100*overlap/len(te):.1f}%) have identical features in train;"
      f" {test_dups.get('exact_duplicates_present', 0):,} exact duplicates inside the official test set (kept).")
integ["unsw_test_rows_overlapping_train"] = [overlap if k == "unsw_nb15" else None for k in DATASETS]

neg = pd.DataFrame([{"dataset": k, "column": c, "negative_rows": n} for k in DATASETS for c, n in LOGS[k]["negative_value_columns"].items()])
if len(neg): print("\nColumns containing negative values (left unchanged):"); display(neg)
integ.to_csv(AUDIT_DIR / "integrity_audit.csv", index=False)
neg.to_csv(AUDIT_DIR / "negative_values.csv", index=False)
(AUDIT_DIR / "cleaning_log.json").write_text(json.dumps(LOGS, indent=2, default=str))

,dataset,rows_in,rows_out,rows_removed_pct,exact_duplicates_removed,conflicting_rows_removed,rows_with_missing_dropped,identifier_cols_dropped,constant_cols_dropped,empty_cols_dropped,cols_with_negatives,features_out,memory_MB
0,bot_iot,911828,903922,0.87,5897,2009,0,12,0,0,0,20,157.8
1,cic_ids2017,2827876,2198595,22.25,546577,82704,0,1,8,0,13,69,746.4
2,unsw_nb15,257673,181600,29.52,45545,30528,0,1,0,0,0,42,38.9


UNSW-NB15: 5,881 of 82,332 test rows (7.1%) have identical features in train; 26,387 exact duplicates inside the official test set (kept).

Columns containing negative values (left unchanged):


,dataset,column,negative_rows
0,cic_ids2017,Flow Duration,115
1,cic_ids2017,Flow Bytes/s,85
2,cic_ids2017,Flow Packets/s,115
3,cic_ids2017,Flow IAT Mean,115
4,cic_ids2017,Flow IAT Max,115
5,cic_ids2017,Flow IAT Min,2890
6,cic_ids2017,Fwd IAT Min,17
7,cic_ids2017,Fwd Header Length,35
8,cic_ids2017,Bwd Header Length,22
9,cic_ids2017,Fwd Header Length.1,35


8169

## 2.6 · Label / target audit

In [ ]:
dist = pd.concat([label_distribution(RAW[k], k, "before") for k in DATASETS] +
                 [label_distribution(CLEAN[k], k, "after") for k in DATASETS], ignore_index=True)
dist.to_csv(AUDIT_DIR / "label_distribution.csv", index=False)

checks, stats = [], {}
for k in DATASETS:
    c, s = label_checks(RAW[k], CLEAN[k], k, CFG[k], RARE_CLASS_THRESHOLD)
    checks.append(c); stats[k] = s
checks = pd.concat(checks, ignore_index=True)
checks.to_csv(AUDIT_DIR / "label_checks.csv", index=False)
display(checks)
display(pd.DataFrame(stats).T)

for k in DATASETS:
    t = dist[dist.dataset == k].pivot_table(index="y_class", columns=["stage", "split"], values="rows", aggfunc="sum", fill_value=0)
    print(f"\n=== {k}"); display(t)

,dataset,check,violations,status,detail
0,bot_iot,no missing labels (clean),0,PASS,
1,bot_iot,"y_binary in {0,1}",0,PASS,
2,bot_iot,y_binary agrees with y_class (clean),0,PASS,
3,bot_iot,raw 'attack' agrees with raw 'category',0,PASS,
4,bot_iot,no class lost during cleaning,0,PASS,
5,bot_iot,classes with < 100 rows,0,PASS,
6,bot_iot,both benign and attack present,0,PASS,
7,cic_ids2017,no missing labels (clean),0,PASS,
8,cic_ids2017,"y_binary in {0,1}",0,PASS,
9,cic_ids2017,y_binary agrees with y_class (clean),0,PASS,


,imbalance_ratio_class,attack_rate,n_classes
bot_iot,188.4,0.9899,5.0
cic_ids2017,169310.2,0.1529,15.0
unsw_nb15,550.7,0.5118,10.0



=== bot_iot


stage,after,before
split,all,all
y_class,,
benign,9143,9543
ddos,298944,301150
dos,298284,299665
reconnaissance,295964,299883
theft,1587,1587



=== cic_ids2017


stage,after,before
split,all,all
y_class,,
benign,1862412,2271320
bot,1397,1956
ddos,128011,128025
dos_goldeneye,10286,10293
dos_hulk,172688,230124
dos_slowhttptest,5228,5499
dos_slowloris,5383,5796
ftp_patator,5931,7935



=== unsw_nb15


stage           after        before       
split            test  train   test  train
y_class                                   
analysis          677    393    677   2000
backdoor          583    403    583   1746
benign          37000  51661  37000  56000
dos              4089   2477   4089  12264
exploits        11132  18417  11132  33393
fuzzers          6062  14602   6062  18184
generic         18871   3904  18871  40000
reconnaissance   3496   6209   3496  10491
shellcode         378   1085    378   1133
worms              44    117     44    130

## 2.7 · Final cleaned-dataset validation & save

In [ ]:
val = pd.concat([validate(CLEAN[k], k, CFG[k], LOGS[k]) for k in DATASETS], ignore_index=True)
val.to_csv(AUDIT_DIR / "validation_report.csv", index=False)
display(val)
failed = val[~val.passed]
if len(failed):
    raise AssertionError(f"{len(failed)} validation checks failed. Nothing was saved. See the table above.")

manifest, schema = {"created_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
                    "module": "2 - data cleaning & audit",
                    "settings": {"DROP_CONFLICTING": DROP_CONFLICTING, "UNSW_DEDUP_TEST": UNSW_DEDUP_TEST,
                                 "RARE_CLASS_THRESHOLD": RARE_CLASS_THRESHOLD},
                    "environment": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__},
                    "datasets": {}}, {}
for k in DATASETS:
    out = OUT_DIR / f"{k}_clean.parquet"
    CLEAN[k].to_parquet(out, index=False)
    feats = feature_cols(CLEAN[k])
    cats = [c for c in feats if isinstance(CLEAN[k][c].dtype, pd.CategoricalDtype)]
    schema[k] = {"features": feats, "numeric_features": [c for c in feats if c not in cats], "categorical_features": cats,
                 "label_columns": [c for c in LABEL_COLS if c in CLEAN[k].columns],
                 "meta_columns": [c for c in META_COLS if c in CLEAN[k].columns],
                 "has_official_split": k == "unsw_nb15"}
    manifest["datasets"][k] = {"input": str((IN_DIR / f"{k}.parquet").relative_to(BASE)),
                               "input_sha256": sha256_file(IN_DIR / f"{k}.parquet"),
                               "output": str(out.relative_to(BASE)), "output_sha256": sha256_file(out),
                               "rows": int(len(CLEAN[k])), "features": len(feats),
                               "attack_rate": stats[k]["attack_rate"], "n_classes": stats[k]["n_classes"]}
    print(f"saved {out.relative_to(BASE)}  {len(CLEAN[k]):,} rows, {len(feats)} features ({len(cats)} categorical)")
(CONFIG_DIR / "feature_schema.json").write_text(json.dumps(schema, indent=2))
(AUDIT_DIR / "module2_manifest.json").write_text(json.dumps(manifest, indent=2))
print("schema   ->", (CONFIG_DIR / "feature_schema.json").relative_to(BASE))
print("manifest ->", (AUDIT_DIR / "module2_manifest.json").relative_to(BASE))

,dataset,check,passed,detail
0,bot_iot,no missing values in features,True,
1,bot_iot,no infinite values,True,
2,bot_iot,no exact duplicate rows (dedup groups),True,0 found
3,bot_iot,features are numeric or categorical,True,
4,bot_iot,no identifier columns left,True,
5,bot_iot,label columns present,True,
6,bot_iot,two target classes present,True,
7,bot_iot,row count matches cleaning log,True,
8,cic_ids2017,no missing values in features,True,
9,cic_ids2017,no infinite values,True,


saved processed/clean/bot_iot_clean.parquet  903,922 rows, 20 features (3 categorical)
saved processed/clean/cic_ids2017_clean.parquet  2,198,595 rows, 69 features (0 categorical)
saved processed/clean/unsw_nb15_clean.parquet  181,600 rows, 42 features (3 categorical)
schema   -> configs/feature_schema.json
manifest -> results/module2/module2_manifest.json


### Module 2 summary

In [ ]:
lines = ["# Module 2 — Data cleaning & audit summary", "",
         f"Run: {manifest['created_utc']}. Settings: drop conflicting duplicates = {DROP_CONFLICTING}; "
         f"deduplicate UNSW-NB15 test = {UNSW_DEDUP_TEST}.", "",
         "| Dataset | Rows in | Rows out | Removed | Exact dups | Conflicting | Features | Attack rate | Classes |",
         "|---|---|---|---|---|---|---|---|---|"]
for _, r in integ.iterrows():
    k = r.dataset
    lines.append(f"| {k} | {r.rows_in:,} | {r.rows_out:,} | {r.rows_removed_pct}% | {r.exact_duplicates_removed:,} | "
                 f"{r.conflicting_rows_removed:,} | {r.features_out} | {stats[k]['attack_rate']:.3f} | {stats[k]['n_classes']} |")
flags = checks[checks.status == "FLAG"]
lines += ["", "## Flags from the label audit", ""]
lines += [f"- **{r.dataset}**: {r.check}: {r.detail or r.violations}" for _, r in flags.iterrows()] or ["- none"]
lines += ["", f"UNSW-NB15: {overlap:,} official-test rows ({100*overlap/len(te):.1f}%) have identical features in train."]
(AUDIT_DIR / "summary.md").write_text("\n".join(lines))
print("\n".join(lines))

# Module 2 — Data cleaning & audit summary

Run: 2026-11-14T13:27:20+00:00. Settings: drop conflicting duplicates = True; deduplicate UNSW-NB15 test = False.

| Dataset | Rows in | Rows out | Removed | Exact dups | Conflicting | Features | Attack rate | Classes |
|---|---|---|---|---|---|---|---|---|
| bot_iot | 911,828 | 903,922 | 0.87% | 5,897 | 2,009 | 20 | 0.990 | 5 |
| cic_ids2017 | 2,827,876 | 2,198,595 | 22.25% | 546,577 | 82,704 | 69 | 0.153 | 15 |
| unsw_nb15 | 257,673 | 181,600 | 29.52% | 45,545 | 30,528 | 42 | 0.512 | 10 |

## Flags from the label audit

- **cic_ids2017**: classes with < 100 rows: infiltration (36), web_attack_sql_injection (21), heartbleed (11)

UNSW-NB15: 5,881 official-test rows (7.1%) have identical features in train.
